In [1]:
### Target: County-Level hourly MW

In [2]:
!pip install shap

In [3]:
!pip install pandas pyarrow

In [4]:
!pip install lightgbm

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns  # for nicer plots
sns.set(style="darkgrid")  # default style
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import lightgbm as lgb

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Read in the data
def read_file(filename):
    directory = '/content/drive/My Drive/210_capstone/final_datasets/train_rolling_windows/'
    location = directory + filename+'.parquet'
    df = pd.read_parquet(location)
    return df

In [8]:
## Read in data
df=read_file('train_window_1')
df.columns

Index(['date', 'county', 'cdd65', 'cdd65_pop', 'cdd75', 'cdd75_pop',
       'cloud_cover_pct_mean', 'cloud_cover_pct_pop', 'dpt_afternoon_k_mean',
       'dpt_afternoon_k_pop', 'dpt_morning_k_mean', 'dpt_morning_k_pop',
       'hdd65', 'hdd65_pop', 'spfh_peak_kgkg_mean', 'spfh_peak_kgkg_pop',
       'tavg_k', 'tmax_k', 'tmax_k_pop', 'tmin_k', 'tmin_k_pop', 'trange_k',
       'wind_low_ms_mean', 'wind_low_ms_pop', 'wind_peak_ms_mean',
       'wind_peak_ms_pop', 'real_data_urma', 'staying_total', 'entering_total',
       'leaving_total', 'real_data_commuting', 'cuml_count', 'cuml_sq_foot',
       'cuml_utility_cap', 'cuml_dc_load', 'real_data_data_centers',
       'electricity_usage', 'public_level_1', 'shared_private_level_1',
       'public_level_2', 'shared_private_level_2', 'public_dc_fast',
       'shared_private_dc_fast', 'total', 'real_data_ev_charging', 'bev',
       'phev', 'fcev', 'real_data_ev_poplution', 'est_median_income',
       'real_data_income', 'total_pop', 'household_

In [9]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42398 entries, 0 to 42397
Data columns (total 70 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    42398 non-null  datetime64[ns]
 1   county                  42398 non-null  object        
 2   cdd65                   42398 non-null  float32       
 3   cdd65_pop               42398 non-null  float32       
 4   cdd75                   42398 non-null  float32       
 5   cdd75_pop               42398 non-null  float32       
 6   cloud_cover_pct_mean    42398 non-null  float32       
 7   cloud_cover_pct_pop     42398 non-null  float32       
 8   dpt_afternoon_k_mean    42398 non-null  float32       
 9   dpt_afternoon_k_pop     42398 non-null  float32       
 10  dpt_morning_k_mean      42398 non-null  float32       
 11  dpt_morning_k_pop       42398 non-null  float32       
 12  hdd65                   42398 non-null  float3

In [10]:
## Fix DataTypes
df['date']=pd.to_datetime(df['date'])

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42398 entries, 0 to 42397
Data columns (total 70 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    42398 non-null  datetime64[ns]
 1   county                  42398 non-null  object        
 2   cdd65                   42398 non-null  float32       
 3   cdd65_pop               42398 non-null  float32       
 4   cdd75                   42398 non-null  float32       
 5   cdd75_pop               42398 non-null  float32       
 6   cloud_cover_pct_mean    42398 non-null  float32       
 7   cloud_cover_pct_pop     42398 non-null  float32       
 8   dpt_afternoon_k_mean    42398 non-null  float32       
 9   dpt_afternoon_k_pop     42398 non-null  float32       
 10  dpt_morning_k_mean      42398 non-null  float32       
 11  dpt_morning_k_pop       42398 non-null  float32       
 12  hdd65                   42398 non-null  float3

## NAN Analysis

In [12]:
# Count nulls per column
null_counts = df.isna().sum().sort_values(ascending=False)

# Show only columns with nulls
null_counts[null_counts > 0]


,0
holiday,41180
shared_private_level_2,26448
real_data_ev_charging,26448
total,26448
public_level_1,26448
public_level_2,26448
public_dc_fast,26448
shared_private_dc_fast,26448
shared_private_level_1,26448


In [13]:
null_percent = (df.isna().mean() * 100).sort_values(ascending=False)

null_percent[null_percent > 0]


,0
holiday,97.127223
shared_private_level_2,62.380301
real_data_ev_charging,62.380301
total,62.380301
public_level_1,62.380301
public_level_2,62.380301
public_dc_fast,62.380301
shared_private_dc_fast,62.380301
shared_private_level_1,62.380301


In [14]:
df[df["date"].isna()].head(10)


,date,county,cdd65,cdd65_pop,cdd75,cdd75_pop,cloud_cover_pct_mean,cloud_cover_pct_pop,dpt_afternoon_k_mean,dpt_afternoon_k_pop,...,occupied,real_data_population,area,year,month,day_of_year,day_of_week,quarter,holiday,is_holiday


In [15]:
null_counts = (df.groupby("county")["electricity_usage"].apply(lambda x: x.isna().sum()).sort_values(ascending=False))

null_counts[null_counts > 0].sort_values(ascending=False)


,electricity_usage
county,


## Feature Store

In [16]:
selected_features = [

    # =========================
    # CATEGORICAL / IDENTIFIERS
    # =========================
    "county",
    #"area",

    # =========================
    # CALENDAR / TIME
    # =========================
    #"year",
    "quarter",
    "month",
    #"day_of_year",
    "day_of_week",
    #"holiday",
    "is_holiday",

    # =========================
    # TEMPERATURE
    # =========================
    #"tavg_k",
    #"tmin_k",
    "tmin_k_pop",
    #"tmax_k",
    "tmax_k_pop",
    #"trange_k",

    # =========================
    # DEGREE DAYS
    # =========================
    #"hdd65",
    "hdd65_pop",
    #"cdd65",
    "cdd65_pop",
    #"cdd75",
    "cdd75_pop",
    "cdd65_pop_roll5",
    "hdd65_pop_roll5",
    "tmax_k_pop_roll5_max",
    # "tmax_k_pop_roll3_mean",
    "tmax_k_pop_roll7_mean",
    # "tmax_k_pop_roll3_max",
    # "tmax_k_pop_roll7_max",

    # =========================
    # DEWPOINT
    # =========================
    #"dpt_afternoon_k_mean",
    "dpt_afternoon_k_pop",
    # "dpt_morning_k_mean",
    # "dpt_morning_k_pop",

    # =========================
    # SPECIFIC HUMIDITY
    # =========================
    #"spfh_peak_kgkg_mean",
    "spfh_peak_kgkg_pop",

    # =========================
    # CLOUD COVER
    # =========================
   # "cloud_cover_pct_mean",
    "cloud_cover_pct_pop",

    # =========================
    # WIND
    # =========================
    #"wind_low_ms_mean",
    #"wind_low_ms_pop",
    #"wind_peak_ms_mean",
    "wind_peak_ms_pop",

    # =========================
    # MOBILITY / COMMUTING
    # =========================
    "staying_total",
    "entering_total",
    "leaving_total",

    # =========================
    # DATA CENTERS
    # =========================
    "cuml_count",
    "cuml_sq_foot",
    "cuml_utility_cap",
    "cuml_dc_load",

    # =========================
    # EV CHARGING INFRASTRUCTURE
    # =========================
    # "public_level_1",
    # "shared_private_level_1",
    # "public_level_2",
    # "shared_private_level_2",
    # "public_dc_fast",
    # "shared_private_dc_fast",
    # "total",

    # =========================
    # EV POPULATION
    # =========================
    "bev",
    "phev",
    "fcev",

    # =========================
    # INCOME
    # =========================
    "est_median_income",

    # =========================
    # DEMOGRAPHICS / HOUSING
    # =========================
    "total_pop",
    # "household_pop",
    # "group_quarters_pop",
    # "total_households",
    # "single_detached",
    # "single_attached",
    # "two_to_four",
    # "five_plus",
    # "mobile_homes",
    # "occupied",

    # =========================
    # DATA QUALITY FLAGS
    # =========================
    # "real_data_urma",
    # "real_data_commuting",
    # "real_data_data_centers",
    # "real_data_ev_charging",
    # "real_data_ev_poplution",
    # "real_data_income",
    # "real_data_population",
]


## Identify Target, Categorical and Cols on full dataset

In [17]:
## Setting up Target / Data Types

target = "electricity_usage"
cat_cols=["county","day_of_week"]

## Light GBM Categorical Variables

## Feature Engineering

In [18]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [19]:
def run_window(train_df, val_df, selected_features, params):

    X_train = train_df[selected_features]
    y_train = train_df[target]
    X_val   = val_df[selected_features]
    y_val   = val_df[target]

    model = lgb.LGBMRegressor(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    preds = model.predict(X_val, num_iteration=model.best_iteration_)
    score = rmse(y_val, preds)

    return model, score, model.best_iteration_

params = dict(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

In [20]:
train_1 = read_file('train_window_1')
train_2 = read_file('train_window_2')
val_1 = read_file('val_window_1')
val_2 = read_file('val_window_2')

print("train_1:", train_1.shape)
print("train_2:", train_2.shape)
print("val_1:", val_1.shape)
print("val_2:", val_2.shape)


train_1: (42398, 70)
train_2: (42398, 70)
val_1: (21170, 70)
val_2: (21170, 70)


In [21]:
# --- rolling feature specs ---
ROLLING_SPECS = [
    # (new_col, source_col, window, agg)
    ("cdd65_pop_roll5",       "cdd65_pop",  5, "sum"),
    ("hdd65_pop_roll5",       "hdd65_pop",  5, "sum"),
    ("tmax_k_pop_roll5_max",  "tmax_k_pop", 5, "max"),
    ("tmax_k_pop_roll3_mean", "tmax_k_pop", 3, "mean"),
    ("tmax_k_pop_roll7_mean", "tmax_k_pop", 7, "mean"),
    ("tmax_k_pop_roll3_max",  "tmax_k_pop", 3, "max"),
    ("tmax_k_pop_roll7_max",  "tmax_k_pop", 7, "max"),
]

def add_rolling_features(df):
    df = df.sort_values(["county", "date"]).copy()
    g = df.groupby("county", sort=False)

    for new_col, src_col, window, agg in ROLLING_SPECS:
        if src_col not in df.columns:
            raise KeyError(f"Missing source column '{src_col}' needed for '{new_col}'")

        df[new_col] = g[src_col].transform(
            lambda x, w=window, a=agg: getattr(x.rolling(w, min_periods=1), a)()
        )
    return df

# --- apply to all your datasets ---
datasets = {
    "train_1": train_1,
    "train_2": train_2,
    "val_1": val_1,
    "val_2": val_2,
}

for name, d in datasets.items():
    datasets[name] = add_rolling_features(d)
    print(f"{name}: added rolling cols -> shape={datasets[name].shape}")

# overwrite your variables (optional convenience)
train_1 = datasets["train_1"]
train_2 = datasets["train_2"]
val_1   = datasets["val_1"]
val_2   = datasets["val_2"]


train_1: added rolling cols -> shape=(42398, 77)
train_2: added rolling cols -> shape=(42398, 77)
val_1: added rolling cols -> shape=(21170, 77)
val_2: added rolling cols -> shape=(21170, 77)


In [22]:
cat_cols = ["county", "day_of_week"]

def set_and_align_categories(train_df, val_df, cat_cols=cat_cols):
    for c in cat_cols:
        if c in train_df.columns and c in val_df.columns:
            train_df[c] = train_df[c].astype("category")
            val_df[c]   = val_df[c].astype("category")
            # align val categories to train categories
            val_df[c] = val_df[c].cat.set_categories(train_df[c].cat.categories)
    return train_df, val_df


In [23]:
train_1, val_1 = set_and_align_categories(train_1, val_1)
train_2, val_2 = set_and_align_categories(train_2, val_2)


In [24]:
model1, rmse1, it1 = run_window(train_1, val_1, selected_features, params)
model2, rmse2, it2 = run_window(train_2, val_2, selected_features, params)

avg_rmse = (rmse1 + rmse2) / 2
print(
    f"""
rmse_window_1: {rmse1:.2f}
rmse_window_2: {rmse2:.2f}
avg_rmse:      {avg_rmse:.2f}
best_iters:    ({it1}, {it2})
"""
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006596 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6462
[LightGBM] [Info] Number of data points in the train set: 42398, number of used features: 30
[LightGBM] [Info] Start training from score 12999.628873
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.026868 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6462
[LightGBM] [Info] Number of data points in the train set: 42398, number of used features: 30
[LightGBM] [Info] Start training from score 13030.571319

rmse_window_1: 4789.93
rmse_window_2: 4979.70
avg_rmse:      4884.82
best_iters:    (211, 52)



In [25]:
mean_load = df["electricity_usage"].mean()

rmse_pct = 100 * avg_rmse / mean_load

print(f"Mean load: {mean_load:,.2f}")
print(f"Avg RMSE:  {avg_rmse:,.2f}")
print(f"RMSE as % of mean load: {rmse_pct:.2f}%")


Mean load: 12,999.63
Avg RMSE:  4,884.82
RMSE as % of mean load: 37.58%


In [26]:
std_load = df["electricity_usage"].std()
print("Std load:", std_load)
print("RMSE / std:", avg_rmse / std_load)


Std load: 28213.739490743283
RMSE / std: 0.17313608900896857


In [27]:
def weighted_rmse(y_true, y_pred, weights):
    return np.sqrt(np.sum(weights * (y_true - y_pred) ** 2) / np.sum(weights))


In [28]:
import numpy as np

def weighted_rmse(y_true, y_pred, weights):
    weights = np.asarray(weights)
    return np.sqrt(np.sum(weights * (y_true - y_pred) ** 2) / np.sum(weights))

preds1 = model1.predict(
    val_1[selected_features],
    num_iteration=model1.best_iteration_
)

w_rmse1 = weighted_rmse(
    val_1["electricity_usage"].values,
    preds1,
    val_1["total_pop"].values
)

print("Population-weighted RMSE (window 1):", w_rmse1)


Population-weighted RMSE (window 1): 15743.89290304855


In [29]:
preds2 = model2.predict(
    val_2[selected_features],
    num_iteration=model2.best_iteration_
)

w_rmse2 = weighted_rmse(
    val_2["electricity_usage"].values,
    preds2,
    val_2["total_pop"].values
)

print("Population-weighted RMSE (window 2):", w_rmse2)
print("Avg population-weighted RMSE:", (w_rmse1 + w_rmse2)/2)


Population-weighted RMSE (window 2): 15392.499693608532
Avg population-weighted RMSE: 15568.196298328541


In [30]:
w_mean_load_1 = np.average(val_1["electricity_usage"].values, weights=val_1["total_pop"].values)
print("Pop-weighted mean load (w1):", w_mean_load_1)
print("Pop-weighted RMSE % (w1):", 100 * w_rmse1 / w_mean_load_1)


Pop-weighted mean load (w1): 71842.76339836429
Pop-weighted RMSE % (w1): 21.914375447600065


In [32]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.4/808.4 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 16.5 MB/s eta 0:00:00


In [35]:
import mlflow
import mlflow.lightgbm
import joblib, os, tempfile
from datetime import datetime

run_ts = datetime.now().strftime('%Y%m%d_%H%M')
model_version = 'v1'
model_name = f'lgbm_{target}_{model_version}_{run_ts}'

MODEL_DIR = '/content/drive/My Drive/210_capstone/models'
PRED_DIR  = '/content/drive/My Drive/210_capstone/predictions'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_DIR,  exist_ok=True)

mlflow.set_tracking_uri(f'sqlite:///{MODEL_DIR}/mlflow.db')
mlflow.set_experiment('climate-feat-lgbm')

with mlflow.start_run(run_name=f'lgbm_{model_version}_{target}_{run_ts}'):

    # ── Params ────────────────────────────────────────────────────────────
    mlflow.log_params(params)
    mlflow.log_param('target', target)
    mlflow.log_param('n_features', len(selected_features))
    mlflow.log_param('best_iter_w1', it1)
    mlflow.log_param('best_iter_w2', it2)
    mlflow.log_param('run_ts', run_ts)

    # ── Raw RMSE (MWh — target is untransformed) ──────────────────────────
    mlflow.log_metric('rmse_w1', rmse1)
    mlflow.log_metric('rmse_w2', rmse2)
    mlflow.log_metric('rmse_avg', avg_rmse)
    mlflow.log_metric('rmse_pct_mean', rmse_pct)

    # ── Population-weighted RMSE ──────────────────────────────────────────
    w_mean_load_2 = np.average(val_2['electricity_usage'].values, weights=val_2['total_pop'].values)
    pop_wtd_rmse_pct_w1 = 100 * w_rmse1 / w_mean_load_1
    pop_wtd_rmse_pct_w2 = 100 * w_rmse2 / w_mean_load_2

    mlflow.log_metric('pop_wtd_rmse_w1', w_rmse1)
    mlflow.log_metric('pop_wtd_rmse_w2', w_rmse2)
    mlflow.log_metric('pop_wtd_rmse_avg', (w_rmse1 + w_rmse2) / 2)
    mlflow.log_metric('pop_wtd_rmse_pct_w1', pop_wtd_rmse_pct_w1)
    mlflow.log_metric('pop_wtd_rmse_pct_w2', pop_wtd_rmse_pct_w2)

    # ── Log models to MLflow ──────────────────────────────────────────────
    mlflow.lightgbm.log_model(model1, artifact_path='lgbm_w1')
    mlflow.lightgbm.log_model(model2, artifact_path='lgbm_w2')

    # ── Feature list as artifact ──────────────────────────────────────────
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
        f.write('\n'.join(selected_features))
        tmp = f.name
    mlflow.log_artifact(tmp, artifact_path='metadata')
    os.unlink(tmp)

    print(f'MLflow run logged — climate-feat-lgbm / {model_name}')
    print(f'  rmse_avg:            {avg_rmse:,.2f} MWh')
    print(f'  rmse_pct_mean:       {rmse_pct:.2f}%')
    print(f'  pop_wtd_rmse_pct w1: {pop_wtd_rmse_pct_w1:.1f}%')
    print(f'  pop_wtd_rmse_pct w2: {pop_wtd_rmse_pct_w2:.1f}%')

# ── Export models + inference artifacts for ensemble ─────────────────────────
joblib.dump(model1, f'{MODEL_DIR}/{model_name}_w1.pkl')
joblib.dump(model2, f'{MODEL_DIR}/{model_name}_w2.pkl')
joblib.dump(selected_features, f'{MODEL_DIR}/{model_name}_features.pkl')

# Save val predictions for ensemble stacking
val_1['preds'] = preds1
val_2['preds'] = preds2

pred_cols = ['county', 'date', 'total_pop', 'electricity_usage', 'preds']

val_1[pred_cols].to_parquet(f'{PRED_DIR}/{model_name}_preds_w1.parquet', index=False)
val_2[pred_cols].to_parquet(f'{PRED_DIR}/{model_name}_preds_w2.parquet', index=False)

print('\nExported for ensemble:')
print(f'  {model_name}_w1.pkl / _w2.pkl')
print(f'  {model_name}_features.pkl')
print(f'  {model_name}_preds_w1.parquet')
print(f'  {model_name}_preds_w2.parquet')

2026/02/23 19:58:53 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/23 19:58:53 INFO mlflow.store.db.utils: Updating database tables
2026/02/23 19:58:58 INFO mlflow.tracking.fluent: Experiment with name 'climate-feat-lgbm' does not exist. Creating a new experiment.
2026/02/23 19:58:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 19:58:59 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/02/23 19:59:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 19:59:07 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle form

MLflow run logged — climate-feat-lgbm / lgbm_electricity_usage_v1_20260223_1958
  rmse_avg:            4,884.82 MWh
  rmse_pct_mean:       37.58%
  pop_wtd_rmse_pct w1: 21.9%
  pop_wtd_rmse_pct w2: 20.9%

Exported for ensemble:
  lgbm_electricity_usage_v1_20260223_1958_w1.pkl / _w2.pkl
  lgbm_electricity_usage_v1_20260223_1958_features.pkl
  lgbm_electricity_usage_v1_20260223_1958_preds_w1.parquet
  lgbm_electricity_usage_v1_20260223_1958_preds_w2.parquet


## OLD Split into Test/Train/Val

In [ ]:
## Rolling Window Cols

# Sort first (critical)
df = df.sort_values(["county", "date"])

df["cdd65_pop_roll5"] = (
    df.groupby("county")["cdd65_pop"]
      .transform(lambda x: x.rolling(5, min_periods=1).sum())
)

df["hdd65_pop_roll5"] = (
    df.groupby("county")["hdd65_pop"]
      .transform(lambda x: x.rolling(5, min_periods=1).sum())
)

df["tmax_k_pop_roll5_max"] = (
    df.groupby("county")["tmax_k_pop"]
      .transform(lambda x: x.rolling(window=5, min_periods=1).max())
)

df["tmax_k_pop_roll3_mean"] = (
    df.groupby("county")["tmax_k_pop"]
      .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

df["tmax_k_pop_roll7_mean"] = (
    df.groupby("county")["tmax_k_pop"]
      .transform(lambda x: x.rolling(window=7, min_periods=1).mean())
)

df["tmax_k_pop_roll3_max"] = (
    df.groupby("county")["tmax_k_pop"]
      .transform(lambda x: x.rolling(3, min_periods=1).max())
)

df["tmax_k_pop_roll7_max"] = (
    df.groupby("county")["tmax_k_pop"]
      .transform(lambda x: x.rolling(7, min_periods=1).max())
)

In [ ]:
# train_end = pd.Timestamp("2018-09-01")
# val_end   = pd.Timestamp("2018-11-01")

# df = df.sort_values(["date"] + [c for c in "county" if c in df.columns])


# train_df = df[df["date"] < train_end]
# val_df   = df[(df["date"] >= train_end) & (df["date"] < val_end)]
# test_df  = df[df["date"] >= val_end]


In [ ]:
# train_shape = train_df.shape
# val_shape = val_df.shape
# test_shape =test_df.shape
# print(f"Shapes:\nTrain:{train_shape}\nVal:{val_shape}\nTest:{test_shape}\n")

In [ ]:
## Remove target cols and date cols (not useful for lightGBM)
df = df.copy() ## copy